# 🦥 BƯỚC 2: UNSLOTH CODING FINE-TUNING PIPELINE (Chạy trên Colab L4 / A100)
Notebook này sử dụng **Unsloth** để huấn luyện / tinh chỉnh mô hình lập trình với dữ liệu code tùy chỉnh (Custom Coding Dataset), tăng tốc độ train 2x-5x và tiết kiệm 80% VRAM nhờ QLoRA.

### 📌 Quy trình:
1. Cài đặt Unsloth và thư viện tương thích.
2. Nạp mô hình đã uncensor từ Bước 1 (trên Google Drive hoặc Hugging Face).
3. Nạp tập dữ liệu huấn luyện coding (Instruction/Input/Output).
4. Huấn luyện QLoRA 4-bit siêu tốc với triton kernels.
5. Merge LoRA adapter vào mô hình gốc và lưu lại Google Drive.

In [ ]:
# @title 1. Cài đặt Unsloth & Dependencies
!nvidia-smi

# Cài đặt Unsloth nhanh chóng
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
# @title 2. Kết nối Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
UNCENSORED_DIR = "/content/drive/MyDrive/ai_coding_models_uncensored"
FINETUNED_DIR = "/content/drive/MyDrive/ai_coding_models_finetuned"
os.makedirs(FINETUNED_DIR, exist_ok=True)
print(f"📁 Thư mục xuất model đã fine-tune: {FINETUNED_DIR}")

In [ ]:
# @title 3. Khởi tạo FastLanguageModel với Unsloth
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096 # Độ dài ngữ cảnh tối đa
DTYPE = None # Tự động chọn (Float16 cho T4/L4)
LOAD_IN_4BIT = True # 4-bit QLoRA

# Chọn đường dẫn model đã uncensor ở bước 1 hoặc từ Hugging Face
MODEL_PATH = "/content/drive/MyDrive/ai_coding_models_uncensored/Qwen2.5-Coder-7B-Heretic-Uncensored"
# Nếu chưa có trên Drive, bạn có thể nạp trực tiếp model Hugging Face:
# MODEL_PATH = "Qwen/Qwen2.5-Coder-7B-Instruct"

print(f"🔄 Đang nạp model từ: {MODEL_PATH}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH if os.path.exists(MODEL_PATH) else "Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

# Cấu hình LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank (16 hoặc 32 cho Code)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth tối ưu 0 dropout
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Cấu hình LoRA Adapter thành công!")

In [ ]:
# @title 4. Chuẩn bị Dataset Coding
from datasets import Dataset, load_dataset

# Template Chat tiêu chuẩn cho Coding
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

# Bạn có thể nạp dataset mẫu từ Hugging Face (ví dụ: 'iamtarun/python_code_instructions_18k_alpaca')
dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split = "train[:2000]")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"📊 Số lượng mẫu huấn luyện: {len(dataset)}")

In [ ]:
# @title 5. Tiến hành Huấn luyện (SFTTrainer)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Tăng lên 300-1000 cho training đầy đủ
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Bắt đầu huấn luyện với Unsloth...")
trainer_stats = trainer.train()
print("🎉 Huấn luyện hoàn tất!")

In [ ]:
# @title 6. Lưu Model đã Fine-tune vào Google Drive (16bit Standalone hoặc GGUF)
OUTPUT_FINETUNED = os.path.join(FINETUNED_DIR, "Qwen2.5-Coder-7B-Finetuned-Final")

print(f"💾 Đang lưu mô hình hợp nhất (Merged 16-bit) sang: {OUTPUT_FINETUNED}...")
model.save_pretrained_merged(OUTPUT_FINETUNED, tokenizer, save_method = "merged_16bit")
print("✅ Lưu hoàn tất! Model này sẵn sàng để đem sang Colab T4 serving!")